<a href="https://colab.research.google.com/github/vischia/atdtf/blob/master/02_trackml_pl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aplicaciones y Tratamiento de Datos en Tecnología y Física
## PL2: From detector hits to particle tracks - the TrackML challenge


(C) Pietro Vischia (Universidad de Oviedo and ICTEA), pietro.vischia@cern.ch

ChatGPT 5.6 Astra has been used as an assistant during the preparation of this exercise.


# From detector hits to particle tracks


**Requirements:** CPU only; aim for 8 GB RAM. Use four labeled events; no GPU or neural-network framework is required.

**The task:** assign each observed detector hit to a reconstructed charged-particle trajectory.
You will build a sparse graph of plausible hit pairs, learn which pairs belong together,
and decode its edges into non-branching tracks. This is a **pattern recognition** exercise.

The core exercise uses central barrel hits to keep the problem small, ignoring the endcap detectors and several other components.

### IMPORTANT NOTE:throughout the exercise, the word *truth* is used to indicate the Monte Carlo truth, that is the true values of a feature, i.e. what in ML is called the *label*

By the end you should be able to:
1. Explain how charged-particle motion motivates tracking features.
2. Build a sparse candidate graph without consulting simulation truth.
3. Train and diagnose an imbalanced edge classifier without event leakage.
4. Convert edge scores into a valid, exclusive hit-to-track assignment.
5. Distinguish candidate coverage, edge metrics, and track-level physics performance.

**Work in order.** Implement the `YOUR CODE HERE` functions; their tests intentionally fail until completed.
Supplied infrastructure is labeled. Do not read the instructor notebook until submission.

**What you need to submit in the tarea:** complete notebook with outputs, your short written answers, a table of results, and the exported CSV on the test event.

## Tracking in particle physics

(see also the resources uploaded for offline consumption)

In an approximately uniform solenoidal magnetic field, a charged particle follows a helix:
its transverse projection is approximately circular. In practical HEP units,
$p_T[\mathrm{GeV}/c] \simeq 0.2998 |q/e| B[\mathrm{T}]R[\mathrm{m}]$.
Scattering, energy loss and nonuniform fields spoil an exact helix. Low-$p_T$ tracks bend more.

TrackML contains simulated hits, not raw detector waveforms. The observable hit coordinates are in **mm**;
truth momenta are in **GeV/c**. Hit IDs are particle identifiers, not row indices: particle IDs can exceed $2^{53}$ (never cast them to floating point) and a truth `particle_id` of zero denotes noise.

**NOTE:** Particle IDs and hit IDs are local to an event; repeated IDs across events do not imply the same object.

We restrict reconstruction using only observed geometry: barrel volumes 8, 13, 17 and
$|\eta_\mathrm{pos}|<1$, where $\eta_\mathrm{pos}=\mathrm{asinh}(z/r)$: we don't consider any other detector element beyond these volumes and regions.
This position-based quantity is not the standard momentum pseudorapidity we usually define in particle physics, particularly for displaced or curved tracks.
We retain noise hits (resulting in more fun :D ). No truth-$p_T$ cut or truth-guided hit subsampling is allowed in the core workflow (i.e. you should not preselect "easy" tracks).


## 0. Prepare the environment and data · supplied infrastructure

Use Python 3.10 or later with NumPy, pandas, SciPy, scikit-learn, matplotlib, threadpoolctl and Jupyter.

The standard `atdtf` conda environment has them all, so in principle you only need to do:

```bash
conda activate atdtf
```

If not (or if you run on colab and it misses packages), you can install the following packages:
```bash
pip install numpy pandas scipy scikit-learn matplotlib threadpoolctl jupyterlab
```

**Data (labelled training sample):** can be downloaded from the [Kaggle trackml website](https://www.kaggle.com/competitions/trackml-particle-identification/data?select=train_sample.zip), accepting the terms of service. In case it doesn't work, there's a copy [here](https://www.hep.uniovi.es/vischia/atdtf/)

Decompress the zipped data into `data/trackml/`

The data loader accepts CSV or CSV.GZ recursively,
but does not extract ZIP archives. Each event needs `event#########-hits.csv`, `-truth.csv` and `-particles.csv`
(or their `.gz` equivalents). Cells and detector alignment tables are not used.

Set `DATA_DIR` below to the extracted folder. The first two discovered events are training, the third validation,
the fourth the held-out test. Freeze this assignment before looking at results. All four come from the labeled
training release: this local test is distinct from Kaggle's unlabeled competition test.

**Offline debugging only:** set `DATA_MODE="toy"` to generate small synthetic tracks in memory.
This mode is clearly labeled throughout. It does not download or substitute for TrackML in the assignment.


In [1]:
from pathlib import Path
import re, json, time, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, precision_recall_curve
from threadpoolctl import threadpool_limits
from IPython.display import display

SEED = 42
DATA_MODE = "trackml"  # "toy" only for a software smoke test
DATA_DIR = Path("data/trackml/train_100_events/")
K = 8
ETA_MAX = 1.0
S_PHI, S_ETA = 0.02, 0.05  # fixed geometric scales, chosen before validation
MAX_TRAIN_EDGES = 250_000
THREADS = 2
SURFACES = [(8, l) for l in (2, 4, 6, 8)] + [(13, l) for l in (2, 4, 6, 8)] + [(17, l) for l in (2, 4)]
SURFACE_RANK = {s: i for i, s in enumerate(SURFACES)}
VERSIONS = {"python": platform.python_version(), "numpy": np.__version__,
            "pandas": pd.__version__, "sklearn": sklearn.__version__}
print(VERSIONS)


{'python': '3.10.21', 'numpy': '2.2.6', 'pandas': '2.3.3', 'sklearn': '1.7.2'}


In [8]:
# Supplied: validated event I/O. Join truth by ID; never assume file-row alignment.
def read_part(prefix, part):
    choices = [Path(str(prefix) + f"-{part}.csv"), Path(str(prefix) + f"-{part}.csv.gz")]
    found = [p for p in choices if p.exists()]
    if len(found) != 1:
        raise FileNotFoundError(f"Need exactly one CSV or CSV.GZ for {prefix}-{part}; found {found}")
    dtype = {"particle_id": "uint64"} if part in ("truth", "particles") else None
    return pd.read_csv(found[0], dtype=dtype)

def load_real(prefix):
    hits, truth, particles = [read_part(prefix, p) for p in ("hits", "truth", "particles")]
    required = {"hits": {"hit_id","x","y","z","volume_id","layer_id","module_id"},
                "truth": {"hit_id","particle_id","weight"},
                "particles": {"particle_id","px","py","pz"}}
    for name, table in [("hits",hits),("truth",truth),("particles",particles)]:
        if not required[name].issubset(table):
            raise ValueError(f"Missing columns in {name}: {required[name] - set(table)}")
    assert hits.hit_id.is_unique and truth.hit_id.is_unique and particles.particle_id.is_unique
    assert set(hits.hit_id) == set(truth.hit_id), "Truth/hit ID mismatch"
    assert np.isfinite(hits[["x","y","z"]].to_numpy()).all()
    assert (truth.weight >= 0).all() and truth.weight.sum() > 0
    assert (truth.loc[truth.particle_id == 0, "weight"] == 0).all(), "Noise must have zero weight"
    return dict(hits=hits, truth=truth, particles=particles)

def discover_prefixes(root):
    files = sorted(set(root.rglob("event*-hits.csv")) | set(root.rglob("event*-hits.csv.gz")))
    prefixes = sorted({str(p).split("-hits.csv")[0] for p in files})
    ids = [int(re.search(r"event(\d+)$", p).group(1)) for p in prefixes]
    if len(ids) != len(set(ids)):
        raise ValueError("Duplicate event IDs under DATA_DIR; keep one copy of each event.")
    if len(prefixes) < 4:
        raise FileNotFoundError(f"Found {len(prefixes)} events in {root.resolve()}; need 4 labeled events.")
    return {eid: p for eid, p in zip(ids, prefixes)}

def toy_event(seed, n=160):
    """Synthetic software fixture; no claim of detector or collision realism."""
    rng = np.random.default_rng(seed)
    radii = np.array([32,72,116,172,260,360,500,660,820,1020.])
    records, labels, pars = [], [], []
    for j in range(n):
        pid = 2**55 + j + 1
        phi0, eta, z0 = rng.uniform(-np.pi,np.pi), rng.uniform(-.85,.85), rng.normal(0,3)
        curvature = rng.choice([-1,1]) * rng.uniform(2e-5,2e-4)
        pt = .0003 / abs(curvature)
        pars.append((pid, pt*np.cos(phi0), pt*np.sin(phi0), pt*np.sinh(eta)))
        for rank, ((vol, layer), r) in enumerate(zip(SURFACES, radii)):
            if rng.random() < .04:  # missing measurements
                continue
            phi = phi0 + curvature*r + rng.normal(0,.0004)
            hid = 10 + 3*len(records)
            records.append((hid, r*np.cos(phi), r*np.sin(phi), z0+r*np.sinh(eta)+rng.normal(0,.1), vol,layer,1))
            labels.append((hid,pid,1.0))
    for (vol,layer), r in zip(SURFACES,radii):
        for _ in range(25):
            phi, eta = rng.uniform(-np.pi,np.pi), rng.uniform(-.95,.95)
            hid = 10 + 3*len(records)
            records.append((hid,r*np.cos(phi),r*np.sin(phi),r*np.sinh(eta),vol,layer,1))
            labels.append((hid,0,0.0))
    h = pd.DataFrame(records,columns=["hit_id","x","y","z","volume_id","layer_id","module_id"])
    t = pd.DataFrame(labels,columns=["hit_id","particle_id","weight"])
    t["particle_id"] = t.particle_id.astype("uint64")
    t["weight"] /= t.weight.sum()
    p = pd.DataFrame(pars,columns=["particle_id","px","py","pz"])
    p["particle_id"] = p.particle_id.astype("uint64")
    return dict(hits=h.sample(frac=1,random_state=seed).reset_index(drop=True),
                truth=t.sample(frac=1,random_state=seed+1).reset_index(drop=True),particles=p)

if DATA_MODE == "trackml":
    PREFIXES = discover_prefixes(DATA_DIR)
    EVENT_IDS = sorted(PREFIXES)[:4]
    def fetch_event(eid):
        return load_real(PREFIXES[eid])
elif DATA_MODE == "toy":
    warnings.warn("SYNTHETIC SOFTWARE CHECK: results are not TrackML performance.")
    EVENT_IDS = [900001,900002,900003,900004]
    def fetch_event(eid):
        return toy_event(eid)
else:
    raise ValueError("DATA_MODE must be 'trackml' or 'toy'")

# ATTENTION: the held out event ID is already set here (in TEST_IDS), but d not load it until E6!!!
# ATTENTION: DO NOT CHANGE THE VALUE OF THESE INDICES
TRAIN_IDS = [1043, 1008]
VAL_IDS = [1064]
TEST_IDS = [1075]

print(DATA_MODE, {"train": TRAIN_IDS, "validation": VAL_IDS, "test": TEST_IDS})


trackml {'train': [1043, 1008], 'validation': [1064], 'test': [1075]}


## E1. Coordinates and a first look

Implement `add_coordinates(hits)` and `wrap_phi(delta)`.
- Return a copy containing $r=\sqrt{x^2+y^2}$, $\phi=\mathrm{atan2}(y,x)$ and $\eta_\mathrm{pos}=\mathrm{asinh}(z/r)$.
- Protect division using $r_\mathrm{safe}=\max(r,10^{-9}\,\mathrm{mm})$.
- Wrap angular differences to $[-\pi,\pi)$; a pair across the branch cut must remain close.
- Preserve input row order and integer identifiers; no truth is needed.

Then inspect the supplied plots. Explain why a straight line from the origin is an approximation,
why low-$p_T$ or displaced tracks are harder, and why random *edge* splits would leak event information.


In [ ]:
def wrap_phi(delta):
    # YOUR CODE HERE
    raise NotImplementedError("E1: wrap angles")

def add_coordinates(hits):
    # YOUR CODE HERE
    raise NotImplementedError("E1: cylindrical coordinates")


In [ ]:
probe = pd.DataFrame({"hit_id":[91,7],"x":[3.,0.],"y":[4.,2.],"z":[5.,0.]})
copy = probe.copy(deep=True)
a = add_coordinates(probe)
assert np.allclose(a.r,[5,2]) and np.allclose(a.eta_pos,[np.arcsinh(1),0])
assert a.hit_id.tolist() == [91,7] and probe.equals(copy)
assert np.isclose(wrap_phi(-np.pi+.01 - (np.pi-.01)), .02)
assert np.isclose(wrap_phi(np.pi), -np.pi)
print("E1 checks passed")

# Supplied selection: depends only on observables and fixed detector geometry.
def select_region(hits):
    h = add_coordinates(hits)
    ranks = np.array([SURFACE_RANK.get((int(v),int(l)),-1) for v,l in zip(h.volume_id,h.layer_id)])
    h["layer_rank"] = ranks
    h = h.loc[(h.layer_rank >= 0) & (h.eta_pos.abs() < ETA_MAX)].copy()
    return h.sort_values("hit_id").reset_index(drop=True)

def region_truth(h, truth):
    return h[["hit_id"]].merge(truth[["hit_id","particle_id","weight"]],
                                on="hit_id",how="left",validate="one_to_one")

raw0 = fetch_event(TRAIN_IDS[0])
h0 = select_region(raw0["hits"])
t0 = region_truth(h0,raw0["truth"])
assert len(h0) > 0 and not t0.isna().any().any()
display(h0.groupby(["volume_id","layer_id"]).agg(n_hits=("hit_id","size"),median_r_mm=("r","median")))
print(f"Region: {len(h0):,}/{len(raw0['hits']):,} hits; noise fraction {(t0.particle_id==0).mean():.3f}")
fig, ax = plt.subplots(1,2,figsize=(11,4))
ax[0].scatter(h0.x,h0.y,s=.4,alpha=.25); ax[0].set_aspect("equal")
ax[0].set(xlabel="x [mm]",ylabel="y [mm]",title="Selected training-event hits")
ax[1].scatter(h0.z,h0.r,s=.4,alpha=.25)
ax[1].set(xlabel="z [mm]",ylabel="r [mm]",title="Barrel geometry and acceptance")
plt.tight_layout(); plt.show()


**Your answer:** Discuss the three questions above in 100–150 words.

*Write here (and make sure you submit the notebook with this answer saved here).*


## E2. Build a sparse candidate graph

An all-pairs graph needs $O(N^2)$ edges. Instead propose at most `k` neighbors on the next configured barrel surface.
For each hit construct
$$u=(\cos\phi/s_\phi,\ \sin\phi/s_\phi,\ \eta_\mathrm{pos}/s_\eta).$$
Euclidean distance in this embedding is periodic in $\phi$ and approximates scaled angular distance for nearby hits.

Implement `build_edges(h, k)`:
1. Loop over consecutive **configured ranks** `0→1, 1→2, ...`; do not silently bridge an empty surface.
2. Fit `NearestNeighbors(algorithm="kd_tree")` on the destination embedding and query the source embedding.
3. Use `min(k, number_of_destination_hits)` neighbors. Skip empty source or destination layers.
4. Return unique `(source_row, destination_row)` integer pairs with shape `(E,2)`, using positional rows of `h`.
5. Return an empty `(0,2)` array if there are no candidates. Never pass truth to this function.

**Complexity:** the output is at most $Nk$; a spatial index avoids explicitly allocating an $N^2$ distance matrix.
Its query runtime is data dependent. Measure it rather than assuming an exact asymptotic speedup.

The supplied diagnostic computes **candidate recall** against every non-noise same-particle pair on adjacent
selected layers, including pairs that the graph omitted. Its denominator is not just generated positive edges.
This is an edge-coverage diagnostic, not track efficiency or a strict upper bound on the weighted track score.


In [ ]:
def embedding(h):
    return np.column_stack([np.cos(h.phi)/S_PHI, np.sin(h.phi)/S_PHI, h.eta_pos/S_ETA])

def build_edges(h, k=K):
    # YOUR CODE HERE; h has a RangeIndex after select_region.
    raise NotImplementedError("E2: sparse adjacent-layer graph")


In [ ]:
# Boundary fixture: a neighbor across +/-pi must be preferred.
small = pd.DataFrame({"phi":[np.pi-.001,-np.pi+.001,0.],"eta_pos":[0.,0.,0.],"layer_rank":[0,1,1]})
assert np.array_equal(build_edges(small,k=1),[[0,1]])
assert build_edges(small.iloc[:1],k=8).shape == (0,2)
assert len(build_edges(small,k=8)) == 2
missing = small.copy(); missing["layer_rank"] = [0,2,2]
assert build_edges(missing,k=8).shape == (0,2)

# Supplied coverage diagnostic, independent of student E3 labels.
def graph_diagnostics(h, truth, edges):
    t = region_truth(h,truth)
    pid = t.particle_id.to_numpy(dtype=np.uint64)
    positive = (pid[edges[:,0]] == pid[edges[:,1]]) & (pid[edges[:,0]] != 0)
    count = pd.DataFrame({"pid":pid,"layer":h.layer_rank.to_numpy()})
    count = count.loc[count.pid != 0].groupby(["pid","layer"]).size().unstack(fill_value=0)
    count = count.reindex(columns=range(len(SURFACES)),fill_value=0).to_numpy(dtype=np.int64)
    denominator = int((count[:,:-1] * count[:,1:]).sum())
    return {"n_hits":len(h),"n_edges":len(edges),"true_adjacent_pairs":denominator,
            "candidate_recall":positive.sum()/denominator if denominator else np.nan,
            "edge_positive_fraction":positive.mean() if len(edges) else np.nan}

coverage_rows = []
for k_value in [2,4,8,16]:
    start = time.perf_counter()
    edges = build_edges(h0,k=k_value)
    assert edges.ndim == 2 and edges.shape[1] == 2 and np.issubdtype(edges.dtype,np.integer)
    assert len(edges) <= len(h0)*k_value
    assert len(np.unique(edges,axis=0)) == len(edges)
    assert (h0.layer_rank.to_numpy()[edges[:,1]] - h0.layer_rank.to_numpy()[edges[:,0]] == 1).all()
    coverage_rows.append({"k":k_value,"build_seconds":time.perf_counter()-start,**graph_diagnostics(h0,raw0["truth"],edges)})
coverage_table = pd.DataFrame(coverage_rows)
display(coverage_table)
e0 = build_edges(h0,K)
print("E2 checks passed")


**Your answer:** Why does increasing k raise cost? Which tracks can still be lost even with a perfect edge classifier? Keep K=8 for the core comparison.

*Write here (and make sure you submit the notebook with this answer saved here).*


## E3. Labels and physics-motivated edge features

Implement two functions. `edge_labels` aligns truth by `hit_id`, then assigns 1 only when the two endpoints
have the same **nonzero** particle ID. Two noise hits are not a true edge.
`edge_features` returns a float32 matrix in the exact feature order below, using only `h` and `edges`.

For each outward pair, let $\Delta r=r_j-r_i$ and use a positive $\Delta r_\mathrm{safe}=\max(\Delta r,10^{-6})$.
Include signed wrapped $\Delta\phi$, signed $\Delta\eta_\mathrm{pos}$, their absolute values,
$\Delta r/1000$, $\Delta z/1000$, $\Delta\phi/\Delta r_\mathrm{safe}$,
$(\Delta z/\Delta r_\mathrm{safe})$, $z_0/1000$ with
$z_0=z_i-r_i\Delta z/\Delta r_\mathrm{safe}$, $r_i/1000$, and the source layer rank.
Distances divided by 1000 are in metres. The slope and origin extrapolation are approximations.
Do not include `particle_id`, hit IDs, truth momentum, truth coordinates, weights or target labels as features.


In [ ]:
FEATURE_NAMES = ["dphi","deta","abs_dphi","abs_deta","dr_m","dz_m",
                 "dphi_dr_per_mm","dz_dr","z0_m","r_src_m","source_layer"]

def edge_labels(h, truth, edges):
    # YOUR CODE HERE
    raise NotImplementedError("E3: align truth and label pairs")

def edge_features(h, edges):
    # YOUR CODE HERE
    raise NotImplementedError("E3: observable-only features")


In [ ]:
# Large, neighboring IDs must remain distinct; shuffled truth must not matter.
id_hits = pd.DataFrame({"hit_id":[31,7,19,42,88]})
id_truth = pd.DataFrame({"hit_id":[88,19,31,42,7],
    "particle_id":np.array([0,2**55+2,2**55+1,0,2**55+1],dtype=np.uint64),"weight":[0,1,1,0,1]})
assert np.array_equal(edge_labels(id_hits,id_truth,np.array([[0,1],[0,2],[3,4]])),[1,0,0])
X0, y0 = edge_features(h0,e0), edge_labels(h0,raw0["truth"],e0)
assert X0.shape == (len(e0),len(FEATURE_NAMES)) and X0.dtype == np.float32
assert np.isfinite(X0).all() and set(np.unique(y0)).issubset({0,1})
assert edge_features(h0,np.empty((0,2),dtype=int)).shape == (0,len(FEATURE_NAMES))
assert np.array_equal(y0,edge_labels(h0,raw0["truth"].sample(frac=1,random_state=2),e0))
assert np.allclose(X0[:,2],abs(X0[:,0])) and np.allclose(X0[:,3],abs(X0[:,1]))
print("E3 checks passed; feature matrix:", X0.shape)


**Your answer:** Explain how the z0 feature might help identify unrelated hits, and give one class of genuine tracks it may disfavor.

*Write here (and make sure you submit the notebook with this answer saved here).*


## E4. Learn which edges belong together

Implement `train_edge_model(X,y)` using a `HistGradientBoostingClassifier` with `max_iter=80`,
`max_leaf_nodes=15`, `learning_rate=0.1`, `l2_regularization=1.0`, `early_stopping=False`, and `random_state=SEED`.
Set sample weights to `n/(2*n_class)` for each class. Disable internal random early-stopping splits so the explicitly
held-out validation event controls model selection. Reject training sets containing only one class.

The supplied training sampler uniformly selects at most 250,000 edges across training events only.
Validation/test candidates remain unsampled. Class weights change the effective class prior: predictions are
ranking scores, not automatically calibrated probabilities. Tune the decision threshold on validation tracks.

Report validation average precision (AP), positive fraction, and a precision–recall curve. Why is accuracy
misleading here? The expected AP of a random ranking is approximately the positive fraction; retain it as a baseline.


In [ ]:
def train_edge_model(X, y):
    # YOUR CODE HERE
    raise NotImplementedError("E4: weighted edge classifier")


In [ ]:
# Supplied event cache and training assembly. Each event keeps its own row-index space.
def make_pack(eid, k=K):
    raw = fetch_event(eid)
    h = select_region(raw["hits"])
    e = build_edges(h,k)
    if len(e) == 0:
        raise ValueError(f"Event {eid}: empty region or graph; check data/selection")
    return dict(eid=eid,raw=raw,h=h,edges=e,X=edge_features(h,e),y=edge_labels(h,raw["truth"],e))

train_packs = [make_pack(eid) for eid in TRAIN_IDS]
val_packs = [make_pack(eid) for eid in VAL_IDS]
# Uniform sampling of a virtual concatenation avoids allocating all training X twice.
lengths = np.array([len(p["y"]) for p in train_packs])
offsets = np.r_[0,np.cumsum(lengths)]
selected = np.sort(np.random.default_rng(SEED).choice(int(offsets[-1]),
                    size=min(MAX_TRAIN_EDGES,int(offsets[-1])),replace=False))
xs,ys = [],[]
for p,lo,hi in zip(train_packs,offsets[:-1],offsets[1:]):
    ix = selected[(selected >= lo)&(selected < hi)] - lo
    xs.append(p["X"][ix]); ys.append(p["y"][ix])
X_train,y_train = np.concatenate(xs),np.concatenate(ys)
start = time.perf_counter()
model = train_edge_model(X_train,y_train)
fit_seconds = time.perf_counter()-start
assert np.array_equal(model.classes_,[0,1])
print(f"Training rows {len(y_train):,}; positive fraction {y_train.mean():.4f}; fit seconds {fit_seconds:.2f}")

def predict_scores(model,X):
    with threadpool_limits(limits=THREADS):
        return model.predict_proba(X)[:,1]

for p in val_packs:
    p["ml_scores"] = predict_scores(model,p["X"])
    assert np.isfinite(p["ml_scores"]).all() and ((p["ml_scores"]>=0)&(p["ml_scores"]<=1)).all()
    if len(np.unique(p["y"])) < 2:
        raise ValueError("Validation edge set needs both classes")
    precision,recall,_ = precision_recall_curve(p["y"],p["ml_scores"])
    ap = average_precision_score(p["y"],p["ml_scores"])
    plt.plot(recall,precision,label=f"event {p['eid']}, AP={ap:.3f}")
    plt.axhline(p["y"].mean(),linestyle="--",color="gray",label="random-ranking baseline")
plt.xlabel("Candidate-edge recall"); plt.ylabel("Candidate-edge precision")
plt.title(f"{DATA_MODE}: validation only"); plt.legend(); plt.show()


**Your answer:** Interpret AP relative to the positive fraction. Why can high AP still produce poor reconstructed tracks?

*Write here (and make sure you submit the notebook with this answer saved here).*


## E5. Turn edge scores into exclusive tracks

Implement `decode_paths(h, edges, scores, threshold)`:
1. Visit edges by decreasing score using a stable sort (`np.argsort(-scores, kind="stable")`).
2. Keep scores **greater than or equal to** the threshold.
3. Accept an edge only if its source has no successor and its destination has no predecessor.
4. Edges always go to the next layer, so the accepted graph is acyclic. Walk forward from every node
   without a predecessor and assign one positive integer `track_id` to each path.
5. Give isolated hits their own track IDs. Return an integer array of length `len(h)`, in row order.

A single ID for all unassigned hits would invent one enormous track. Do not do this.
Layer monotonicity implies at most one hit per configured layer in a path; this is an intentional approximation
that loses overlapping-module hits and curling particles. This greedy algorithm is not a global optimum.
No truth is available to the decoder.


In [ ]:
def decode_paths(h, edges, scores, threshold):
    # YOUR CODE HERE
    raise NotImplementedError("E5: greedy exclusive path cover")


In [ ]:
dh = pd.DataFrame({"layer_rank":[0,1,1,2,0]})
de = np.array([[0,1],[0,2],[1,3],[2,3]])
ds = np.array([.9,.8,.7,.95])
dt = decode_paths(dh,de,ds,.5)
assert dt[0] == dt[1] and dt[2] == dt[3] and dt[0] != dt[2]
assert dt[4] not in dt[:4]
assert len(set(decode_paths(dh,de,ds,1.1))) == len(dh)
assert len(set(decode_paths(dh,np.empty((0,2),dtype=int),np.array([]),.5))) == len(dh)
assert decode_paths(dh,de,ds,.95)[2] == decode_paths(dh,de,ds,.95)[3]
print("E5 checks passed")


**Your answer:** Construct a small graph where greedy selection loses to a globally better set of links; propose one improvement that uses no truth.

*Write here (and make sure you submit the notebook with this answer saved here).*


## E6. Evaluate reconstructed tracks

For each reconstructed track $R$, let $P$ be its most frequent truth particle, $m=|R\cap P|$,
$n_R=|R|$, and $n_P=|P|$ in the **evaluated hit collection**. A track contributes only when
$$m/n_R>1/2\quad\text{and}\quad m/n_P>1/2.$$
Its contribution is the sum of the truth weights of the overlapping hits divided by total evaluated truth weight.
Sum over accepted tracks. Both inequalities are **strict**. Noise has zero truth weight and cannot contribute;
noise hits still count in reconstructed-track sizes. Do not reward all hits of a matched truth particle.

Implement `weighted_track_score(truth, submission)`. Require one prediction for every truth hit, no extra hits,
and unique hit IDs. For each track choose the most frequent particle; a tie cannot pass the strict purity test.
Return 0 for zero total weight. This reproduces the TrackML weighting rule on the supplied collection.

We report two distinct numbers:
- **Regional score:** truth restricted to selected hits; completeness counts and total weight are recomputed in the region.
  This is an educational, easier target; it is not comparable to the competition leaderboard.
- **Full-event score:** retain original full-event truth, embed regional predictions, and give every excluded hit a unique
  singleton ID. This has the original denominator and completeness counts, and exposes the limited scope.
  Singleton truth particles can contribute, so report the all-singletons baseline too.

Choose thresholds using the **validation regional score**, then freeze them. Evaluate the local held-out test only once.


In [ ]:
def weighted_track_score(truth, submission):
    # YOUR CODE HERE
    raise NotImplementedError("E6: double-majority weighted score")


In [ ]:
# Strict-half, noise, partial overlap, shuffled rows and denominator tests.
st = pd.DataFrame({"hit_id":np.arange(7),"particle_id":[11,11,11,22,22,22,0],
                   "weight":[1/6]*6+[0.]})
def scored(ids,t=st):
    return weighted_track_score(t,pd.DataFrame({"hit_id":t.hit_id,"track_id":ids}))
assert np.isclose(scored([1,1,1,2,2,2,3]),1.)
assert np.isclose(scored([1,1,1,1,1,1,1]),0.)
assert np.isclose(scored([1,1,2,3,3,4,5]),4/6)
assert np.isclose(scored(list(range(7))),0.)
half = pd.DataFrame({"hit_id":[1,2],"particle_id":[9,9],"weight":[.2,.8]})
assert scored([1,2],half) == 0
unequal = pd.DataFrame({"hit_id":[1,2,3],"particle_id":[9,9,9],"weight":[.1,.2,.7]})
assert np.isclose(scored([1,1,2],unequal),.3)
zero = st.copy(); zero["weight"] = 0.
assert scored([1]*7,zero) == 0.
print("E6 scoring checks passed")

# Supplied evaluation and geometric baseline. Baseline uses exactly the same graph and decoder.
def geometric_scores(X):
    return np.exp(-.5*((X[:,0]/S_PHI)**2+(X[:,1]/S_ETA)**2))

def as_submission(h,ids):
    return pd.DataFrame({"hit_id":h.hit_id.to_numpy(),"track_id":np.asarray(ids,dtype=np.int64)})

def full_submission(pack,regional_ids):
    full = pack["raw"]["hits"][["hit_id"]].copy()
    full["track_id"] = np.arange(1,len(full)+1,dtype=np.int64) + int(np.max(regional_ids,initial=0))
    mapping = pd.Series(regional_ids,index=pack["h"].hit_id)
    mask = full.hit_id.isin(mapping.index)
    full.loc[mask,"track_id"] = full.loc[mask,"hit_id"].map(mapping).to_numpy(dtype=np.int64)
    return full

def evaluate(pack,scores,threshold):
    start = time.perf_counter()
    ids = decode_paths(pack["h"],pack["edges"],scores,threshold)
    seconds = time.perf_counter()-start
    reg_truth = region_truth(pack["h"],pack["raw"]["truth"])
    return {"region_score":weighted_track_score(reg_truth,as_submission(pack["h"],ids)),
            "full_score":weighted_track_score(pack["raw"]["truth"],full_submission(pack,ids)),
            "decode_seconds":seconds,"n_tracks":len(np.unique(ids))},ids

ML_GRID = np.r_[.05,.1,.2,.3,.4,.5,.6,.7,.8,.9,.95,.98,.995]
GEO_GRID = np.r_[1e-6,1e-4,.001,.01,.05,.1,.2,.4,.6,.8,.95]
val_rows = []
for method,grid in [("geometry",GEO_GRID),("ml",ML_GRID)]:
    for threshold in grid:
        event_scores = []
        for p in val_packs:
            s = geometric_scores(p["X"]) if method == "geometry" else p["ml_scores"]
            ids = decode_paths(p["h"],p["edges"],s,threshold)
            event_scores.append(weighted_track_score(region_truth(p["h"],p["raw"]["truth"]),as_submission(p["h"],ids)))
        val_rows.append({"method":method,"threshold":float(threshold),"mean_region_score":np.mean(event_scores)})
validation_table = pd.DataFrame(val_rows)
# Break ties in favor of the higher threshold, by a predeclared deterministic rule.
best = validation_table.sort_values(["mean_region_score","threshold"],ascending=False).drop_duplicates("method")
THRESHOLDS = dict(zip(best.method,best.threshold))
display(best)
for method,group in validation_table.groupby("method"):
    plt.plot(group.threshold,group.mean_region_score,"o-",label=method)
plt.xlabel("Score threshold"); plt.ylabel("Validation regional score"); plt.legend(); plt.show()


### Validation experiment before opening the test set

Complete E7's feature ablation here so the test remains sealed. Remove `z0_m`, retrain on the same sampled rows,
then tune only its threshold on validation. The full model remains the predeclared core model;
the ablation is a diagnostic, not permission to choose using the test results.

## E7a — implement `run_ablation`

returning a fitted model, the column indices kept, and the best threshold. Use the same classifier settings, class weights and ML threshold grid.
Resolve ties toward the higher threshold. Do not access `TEST_IDS` or test truth.


In [ ]:
def run_ablation(X_train,y_train,val_packs):
    # YOUR CODE HERE
    raise NotImplementedError("E7: remove z0, retrain, validate")


In [ ]:
ablation_model,ABLATION_COLUMNS,ablation_threshold = run_ablation(X_train,y_train,val_packs)
assert len(ABLATION_COLUMNS) == len(FEATURE_NAMES)-1
assert FEATURE_NAMES.index("z0_m") not in ABLATION_COLUMNS
assert ablation_threshold in ML_GRID
THRESHOLDS["ml_without_z0"] = ablation_threshold
print("Frozen thresholds:",THRESHOLDS)

# STOP: record settings before running the next cell. No retuning after test evaluation.
FROZEN_CONFIG = {"seed":SEED,"mode":DATA_MODE,"train_ids":TRAIN_IDS,"validation_ids":VAL_IDS,
    "test_ids":TEST_IDS,"k":K,"eta_max":ETA_MAX,"surfaces":SURFACES,"s_phi":S_PHI,"s_eta":S_ETA,
    "max_train_edges":MAX_TRAIN_EDGES,"features":FEATURE_NAMES,"thresholds":THRESHOLDS.copy(),"versions":VERSIONS}
print(json.dumps(FROZEN_CONFIG,indent=2))


In [ ]:
# Supplied final evaluation: now and only now load the held-out labeled event.
test_packs = [make_pack(eid) for eid in TEST_IDS]
results, test_assignments = [],{}
for p in test_packs:
    methods = {"all_singletons":(np.zeros(len(p["edges"])),1.1),
               "geometry":(geometric_scores(p["X"]),THRESHOLDS["geometry"]),
               "ml":(predict_scores(model,p["X"]),THRESHOLDS["ml"]),
               "ml_without_z0":(predict_scores(ablation_model,p["X"][:,ABLATION_COLUMNS]),THRESHOLDS["ml_without_z0"])}
    diagnostics = graph_diagnostics(p["h"],p["raw"]["truth"],p["edges"])
    for method,(scores,threshold) in methods.items():
        metrics,ids = evaluate(p,scores,threshold)
        pred = scores >= threshold
        tp = int(np.sum(pred & (p["y"] == 1)))
        results.append({"mode":DATA_MODE,"event_id":p["eid"],"method":method,"threshold":threshold,
            **metrics,**diagnostics,
            "edge_AP":average_precision_score(p["y"],scores),
            "edge_precision":tp/max(int(pred.sum()),1),
            "edge_recall_within_graph":tp/max(int(p["y"].sum()),1)})
        test_assignments[(p["eid"],method)] = ids
results_table = pd.DataFrame(results)
display(results_table.round(4))
print("Edge precision/recall above use the threshold BEFORE exclusive decoding.")


### Physics diagnostic · supplied

Measure particle reconstruction efficiency versus truth $p_T$. A particle is eligible if it has at least
five selected hits across five distinct selected layers. A particle counts as recovered when one predicted
track contains a strict majority of its selected hits and has strict majority purity.
This is an **unweighted, selected-region particle metric**, separate from the weighted score.
Truth momentum enters only this evaluation, never the features or graph. Bin counts are printed;
empty bins remain NaN, not zero. One held-out event cannot establish robust generalization.


In [ ]:
def particle_efficiency(pack,ids):
    h,t = pack["h"],region_truth(pack["h"],pack["raw"]["truth"])
    a = t.assign(track_id=ids,layer_rank=h.layer_rank.to_numpy())
    nrec = a.groupby("track_id").size()  # includes noise
    stats = a[a.particle_id != 0].groupby("particle_id").agg(n=("hit_id","size"),layers=("layer_rank","nunique"))
    ov = a[a.particle_id != 0].groupby(["track_id","particle_id"]).size().rename("m").reset_index()
    good = (2*ov.m > ov.track_id.map(nrec)) & (2*ov.m > ov.particle_id.map(stats.n))
    recovered = set(ov.loc[good,"particle_id"])
    eligible = stats[(stats.n>=5)&(stats.layers>=5)].reset_index()
    eligible = eligible.merge(pack["raw"]["particles"][["particle_id","px","py"]],on="particle_id",validate="one_to_one")
    eligible["pt"] = np.hypot(eligible.px,eligible.py)
    eligible["recovered"] = eligible.particle_id.isin(recovered)
    eligible["pt_bin"] = pd.cut(eligible.pt,[0,.5,1,2,5,10,np.inf],right=False)
    out = eligible.groupby("pt_bin",observed=False).agg(n=("recovered","size"),recovered=("recovered","sum"))
    out["efficiency"] = out.recovered/out.n.replace(0,np.nan)
    return out

p = test_packs[0]
eff = particle_efficiency(p,test_assignments[(p["eid"],"ml")])
display(eff)
eff.efficiency.plot.bar(rot=30,ylim=(0,1),ylabel="Selected-particle efficiency",xlabel="Truth pT [GeV/c]",
                        title=f"{DATA_MODE}: eligible particles in event {p['eid']}")
plt.tight_layout(); plt.show()

# Inspect a fixed central hit patch; line segments are associations, not fitted trajectories.
h = p["h"]
ids = test_assignments[(p["eid"],"ml")]
mask = (abs(h.phi) < .25) & (abs(h.eta_pos) < .25)
fig,ax = plt.subplots(figsize=(7,5))
ax.scatter(h.loc[mask,"x"],h.loc[mask,"y"],s=5,c="lightgray")
shown = 0
for tid in np.unique(ids[mask])[:30]:
    rows = np.flatnonzero(ids == tid)
    if len(rows) < 3:
        continue
    rows = rows[np.argsort(h.layer_rank.to_numpy()[rows])]
    ax.plot(h.x.to_numpy()[rows],h.y.to_numpy()[rows],".-",alpha=.65,lw=1)
    shown += 1
ax.set(xlabel="x [mm]",ylabel="y [mm]",title=f"Up to 30 tracks touching a fixed patch ({shown} drawn)")
ax.set_aspect("equal"); plt.show()


**Your answer:** Compare regional and full-event scores, and interpret the pT dependence without overstating statistical precision.

*Write here (and make sure you submit the notebook with this answer saved here).*


## E7b. Controlled comparison and short report

Use the already-frozen geometry, full-ML and no-z0 runs. Complete a 300–500 word discussion:
1. Summarize the E2 recall/cost tradeoff for k=2,4,8,16 on the training event.
2. Compare geometry and ML on the same held-out candidates and decoder. Include AP, regional score,
   full-event score and decode time. Decode time excludes graph construction, features and inference;
   do not present it as end-to-end runtime.
3. Compare full ML with the no-z0 ablation. Report a neutral or negative result honestly; z0 can be
   partially reconstructed from other features. An ablation is not proof of causality.
4. Inspect at least one failure using truth **after** final evaluation: merged particles, a split trajectory,
   missed candidates or a greedy conflict. Explain the mechanism.
5. Propose one next step and a validation experiment. Do not change the frozen system based on this test.

**Answer template**
- Candidate coverage and cost: …
- Geometry versus learned edges: …
- Ablation result and interpretation: …
- One observed failure: …
- Limitation, next step and validation design: …

**Rubric:** E1 correctness/physics 6+4; E2 graph/edge cases/coverage interpretation 10+5+5;
E3 labels/features 5+5; E4 training/metrics/interpretation 7+4+4; E5 constraints/tests/critique 10+5+5;
E6 scoring/held-out protocol/physics metrics 7+4+4; E7 controlled comparison/report 5+5.
The tests are checks of important contracts, not a complete autograder. Add a small test if your chosen implementation
introduces an additional corner case. No credit for performance numbers obtained through truth-based candidate construction.


In [ ]:
# Supplied failure inspection table. Select a track from this table for the report.
inspection = region_truth(p["h"],p["raw"]["truth"]).assign(track_id=ids)
track_review = inspection.groupby("track_id").agg(n_hits=("hit_id","size"),
    n_nonzero_particles=("particle_id",lambda s:s[s!=0].nunique()))
display(track_review[track_review.n_hits>=3].sort_values(["n_nonzero_particles","n_hits"],ascending=False).head(12))
# Optional: inspect all truth associations of one chosen track.
# display(inspection[inspection.track_id == CHOSEN_TRACK_ID])


## Submission and reproducibility

Finish the written answers, then restart the kernel and run all cells. The scaffold is intentionally incomplete;
a complete student notebook should execute without `NotImplementedError`. Keep event IDs, versions and frozen
settings with the result table. Do not include private Kaggle credentials or redistribute data contrary to its terms.

The next cell writes one CSV with columns `event_id, hit_id, track_id` for every hit in each local test event,
plus a results table and configuration. Track IDs need only be unique within an event. This has the challenge's
column structure, but uses our labeled local test events; it is **not** a competition submission.
File names explicitly distinguish synthetic output from TrackML output.


In [ ]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
frames = []
for pack in test_packs:
    frame = full_submission(pack,test_assignments[(pack["eid"],"ml")])
    frame.insert(0,"event_id",pack["eid"])
    assert len(frame) == len(pack["raw"]["hits"]) and frame.hit_id.is_unique
    assert frame.track_id.notna().all() and (frame.track_id > 0).all()
    frames.append(frame)
submission = pd.concat(frames,ignore_index=True)
assert not submission.duplicated(["event_id","hit_id"]).any()
submission.to_csv(OUTPUT_DIR/f"{DATA_MODE}_local_test_tracks.csv",index=False)
results_table.to_csv(OUTPUT_DIR/f"{DATA_MODE}_results.csv",index=False)
(OUTPUT_DIR/f"{DATA_MODE}_config.json").write_text(json.dumps(FROZEN_CONFIG,indent=2))
print("Saved outputs in",OUTPUT_DIR.resolve())
